# Modelado de Fatiga con Red Híbrida CNN + LSTM

Este notebook contiene la explicación teórica, la revisión de literatura científica, la arquitectura detallada, y la implementación paso a paso de una red **CNN-LSTM híbrida en PyTorch** (empleando convoluciones 1D conectadas a nuestra LSTM manual) para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

La arquitectura híbrida **CNN-LSTM** combina dos de las estructuras de redes neuronales más potentes para el modelado de señales fisiológicas temporales:
1. **Capa Convolucional 1D (CNN):** Funciona como un extractor de características a corto plazo, aplicando filtros deslizantes sobre el eje temporal de la señal para capturar patrones locales (como los picos de las ondas del ECG o variaciones transitorias de la EDA).
2. **Capa Recurrente (Custom LSTM):** Modela las dependencias secuenciales a largo plazo de los patrones extraídos por la CNN.

### Formulación Convolucional 1D y Downsampling

Dado un tensor de secuencia fisiológica $X \in \mathbb{R}^{B \times L_{in} 	imes C_{in}}$, la convolución 1D del canal de salida $j$ en el instante $t$ se calcula como:
$$y_{j, t} = b_j + \sum_{i=1}^{C_{in}} K_{j, i} \star X_{i, t:t+k-1}$$
Donde $K_{j, i}$ es el kernel convolucional de tamaño $k$ y $b_j$ es el sesgo.

Posteriormente se aplica pooling temporal (**MaxPool1d** o **AvgPool1d**) con un tamaño de ventana $p$ y stride $s$, reduciendo el tamaño de la secuencia a:
$$L_{out} = \lfloor \frac{L_{in} - p}{s} \rfloor + 1$$

### ¿Por qué es eficiente el acoplamiento?

El procesamiento recurrente es secuencial paso a paso ($O(L_{in})$) y no es paralelizable a nivel de pasos temporales. Al aplicar Convolución 1D y Pooling, la secuencia original de longitud $L_{in}$ (por ejemplo, 128 timesteps) se comprime temporalmente a una longitud reducida $L_{out}$ (por ejemplo, 64 timesteps) que representa características complejas compactas.

Esto hace que la LSTM procese la mitad de pasos temporales, disminuyendo a la mitad la longitud del grafo de computación para la retropropagación en el tiempo. Como resultado, el entrenamiento es sustancialmente más rápido y se mitiga el problema del desvanecimiento del gradiente.

---

### Diagrama de Flujo de Tensores (Mermaid)

```mermaid
graph TD
    subgraph Entrada
        in["Secuencia Fisiológica: (B, seq_len, features)"]
    end

    subgraph "Alineación de Canales (Transpose)"
        tr1["Transpose(1, 2)"]
        out_tr1["Tensor para CNN: (B, features, seq_len)"]
        in --> tr1
        tr1 --> out_tr1
    end

    subgraph "Capa Convolucional 1D (nn.Conv1d)"
        conv["Filtros Convolucionales 1D (kernel_size=3)"]
        act["Activación No Lineal: nn.ReLU()"]
        out_conv["Tensor local: (B, conv_channels, seq_len)"]
        out_tr1 --> conv
        conv --> act
        act --> out_conv
    end

    subgraph "Submuestreo Temporal (nn.MaxPool1d)"
        pool["Max Pooling 1D (pool_size=2)"]
        out_pool["Secuencia comprimida: (B, conv_channels, new_seq_len)"]
        out_conv --> pool
        pool --> out_pool
    end

    subgraph "Re-alineación Temporal (Transpose)"
        tr2["Transpose(1, 2)"]
        out_tr2["Tensor para LSTM: (B, new_seq_len, conv_channels)"]
        out_pool --> tr2
        tr2 --> out_tr2
    end

    subgraph "Lógica Recurrente (CustomLSTM)"
        lstm["CustomLSTM (2 capas, dropout=0.2)"]
        out_lstm["Último Estado Oculto (h_n): (B, hidden_size)"]
        out_tr2 --> lstm
        lstm --> out_lstm
    end

    subgraph "Predicción (nn.Linear)"
        fc["Capa Lineal Reguladora"]
        pred["Predicción de Fatiga: (B, 2)"]
        out_lstm --> fc
        fc --> pred
    end
```

---

### Citas Bibliográficas Científicas

* **Shi, X., Chen, Z., Wang, H., Yeung, D. Y., Wong, W. K., & Woo, W. C. (2015).** *Convolutional LSTM Network: A Machine Learning Approach for Precipitation Nowcasting*. Advances in Neural Information Processing Systems (NeurIPS), 28, 802-810. [Enlace al Paper](https://papers.nips.cc/paper/2015/hash/07563a3fe3bbe7e3ba84431ad9d055af-Abstract.html)
* **Staudemeyer, R. C., & Morris, C. (2019).** *Understanding LSTM - a tutorial into Long Short-Term Memory recurrent neural networks*. arXiv preprint arXiv:1909.09586. [Enlace al Paper](https://arxiv.org/abs/1909.09586).

In [1]:
# SETUP e IMPORTACIONES
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
from fatigueset.models import CustomCNNLSTMRegressor, FatigueSequenceDataset
from fatigueset.models.rnn import _prepare_target_table, _merge_raw_streams, _build_sequences

print("[OK] Imports completados y path configurado.")
print(f"Dispositivo actual: {'cuda' if torch.cuda.is_available() else 'cpu'}")

[OK] Imports completados y path configurado.
Dispositivo actual: cuda


## 2. Configuración del Pipeline y Construcción de Secuencias

Cargamos los datos fisiológicos crudos (ECG, EDA, Resp, EEG) desde el dataset `fatigueset`, alineamos los streams temporales mediante `merge_asof` e interpolamos valores perdidos para construir tensores de secuencias temporales coherentes.

In [2]:
# Configuración del dataset y pipeline
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset...")
raw = pipeline.cargar_dataset(verbose=False)

print("Preparando targets del dataframe ML...")
df_ml = pipeline.construir_dataset_ml(raw)
df_targets = _prepare_target_table(df_ml)

print("Combinando streams fisiológicos crudos (Chest y Wrist)...")
df_raw = _merge_raw_streams(raw)

# Parámetros de ventanas de secuencia temporal
seq_len = 128
step = 32

print(f"Construyendo secuencias de tamaño={seq_len} y paso={step}...")
X_arr, y_arr, groups, feature_cols = _build_sequences(
    df_raw=df_raw,
    df_targets=df_targets,
    seq_len=seq_len,
    step=step
)

print(f"[OK] Dimensiones de tensores construidos:")
print(f"  - X: {X_arr.shape} (Número de ventanas x seq_len x features)")
print(f"  - y: {y_arr.shape} (Número de ventanas x 2 targets)")
print(f"  - Columnas de sensores: {len(feature_cols)}")

Cargando dataset...


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


Preparando targets del dataframe ML...


Combinando streams fisiológicos crudos (Chest y Wrist)...


Construyendo secuencias de tamaño=128 y paso=32...
[OK] Dimensiones de tensores construidos:
  - X: (1306, 128, 23) (Número de ventanas x seq_len x features)
  - y: (1306, 2) (Número de ventanas x 2 targets)
  - Columnas de sensores: 23


## 3. División de Datos por Participante (Group Split) y DataLoaders

Para garantizar la rigurosidad científica y evitar la fuga de información (data leakage), separamos el dataset de tal manera que el sujeto utilizado para validar/probar nunca esté presente en el conjunto de entrenamiento.

In [3]:
# Hacemos una partición de prueba donde el participante '01' se reserva para validación
train_idx = np.where(groups != '01')[0]
val_idx = np.where(groups == '01')[0]

X_train, y_train = X_arr[train_idx], y_arr[train_idx]
X_val, y_val = X_arr[val_idx], y_arr[val_idx]

train_dataset = FatigueSequenceDataset(X_train, y_train)
val_dataset = FatigueSequenceDataset(X_val, y_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Train samples: 1184
Validation samples: 122


## 4. Inicialización del Regresor Híbrido CNN + LSTM

Instanciamos nuestro modelo regresor `CustomCNNLSTMRegressor` usando el número de características predictivas fisiológicas detectadas. El modelo extrae patrones locales con Conv1D y alimenta la CustomLSTM.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = len(feature_cols)
conv_channels = 64
hidden_size = 64
num_layers = 2
dropout = 0.2

model = CustomCNNLSTMRegressor(
    input_size=input_size,
    conv_channels=conv_channels,
    kernel_size=3,
    pool_size=2,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    output_size=2
).to(device)

print(model)

CustomCNNLSTMRegressor(
  (conv): Conv1d(23, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (lstm): CustomLSTM(
    (layers): ModuleList(
      (0-1): 2 x CustomLSTMCell()
    )
    (dropout_layer): Dropout(p=0.2, inplace=False)
  )
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


## 5. Entrenamiento Corto de Validación (Sanity Check)

Ejecutamos un entrenamiento corto de 5 épocas sobre el conjunto de datos para verificar la estabilidad de las operaciones de tensores en las celdas híbridas y del motor de optimización.

In [5]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("Iniciando mini-entrenamiento...")

for epoch in range(1, epochs + 1):
    # Modo entrenamiento
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        
        # Gradient clipping para evitar inestabilidad recurrente
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Modo validación
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    
    print(f"Epoch {epoch}/{epochs} - Train Loss (MSE): {avg_train_loss:.6f} - Val Loss (MSE): {avg_val_loss:.6f}")

print("[OK] Entrenamiento corto finalizado correctamente.")

Iniciando mini-entrenamiento...


Epoch 1/5 - Train Loss (MSE): 1242.564811 - Val Loss (MSE): 1118.957214


Epoch 2/5 - Train Loss (MSE): 1029.760849 - Val Loss (MSE): 920.919189


Epoch 3/5 - Train Loss (MSE): 896.690144 - Val Loss (MSE): 776.988586


Epoch 4/5 - Train Loss (MSE): 793.247859 - Val Loss (MSE): 652.574715


Epoch 5/5 - Train Loss (MSE): 701.741204 - Val Loss (MSE): 543.631584
[OK] Entrenamiento corto finalizado correctamente.


## 6. Serialización del Modelo de Fatiga

Persistimos el estado del modelo en la carpeta centralizada de modelos del proyecto `/models/` bajo la categoría correspondiente.

In [6]:
output_dir = Path.cwd().parent / "models" / "deep_learning"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "cnn_lstm_fatigue_notebook.pt"
torch.save(model.state_dict(), model_path)

print(f"[OK] Modelo guardado exitosamente en: {model_path}")

[OK] Modelo guardado exitosamente en: C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\deep_learning\cnn_lstm_fatigue_notebook.pt
